In [6]:
import zipfile
import os

zip_file_path = '/content/merged_poultry_dataset.zip'
extraction_path = '/content/merged_poultry_dataset/' # Directory to extract to

# Create the directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"'{zip_file_path}' unzipped to '{extraction_path}'")
print("Contents of the extracted directory:")
print(os.listdir(extraction_path))

'/content/merged_poultry_dataset.zip' unzipped to '/content/merged_poultry_dataset/'
Contents of the extracted directory:
['merged_poultry_dataset']


In [2]:
# # Cell 1 — Environment & reproducibility
# !pip -q install -U ultralytics
import ultralytics, torch, random, numpy as np, os
ultralytics.checks()

SEED = 0
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

WARNING Multiple Ultralytics installations detected. The `yolo` command uses: C:\dev\poultry-vision\ultralytics, but current session imports from: C:\dev\poultry-vision\.venv\Lib\site-packages\ultralytics. This may cause version conflicts. Install your local copy in editable mode with 'pip install -e C:\dev\poultry-vision\.venv\Lib\site-packages' to avoid issues. See https://docs.ultralytics.com/quickstart/
Ultralytics 8.4.66  Python-3.13.14 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
Setup complete  (12 CPUs, 31.1 GB RAM, 319.6/464.8 GB disk)
CUDA: True | NVIDIA GeForce RTX 5070


In [ ]:
# Cell 2 — Write a corrected data.yaml (the uploaded one points to a Windows path)
import yaml, os

DATA_ROOT = "/content/merged_poultry_dataset/merged_poultry_dataset"  # your unzip target
assert os.path.isdir(DATA_ROOT), f"Not found: {DATA_ROOT}"

data_cfg = {
    "path":  DATA_ROOT,
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc": 3,
    "names": ["feeder", "hen", "waterer"],
}
DATA_YAML = os.path.join(DATA_ROOT, "data_colab.yaml")
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print(open(DATA_YAML).read())

In [4]:
# Cell 2 — Write a corrected data.yaml for the local Windows dataset
import yaml, os

# Relative to the repo root (c:\dev\poultry-vision). Use forward slashes — they work fine on Windows.
DATA_ROOT = os.path.abspath("../dataset/segment/merged_poultry_dataset")
assert os.path.isdir(DATA_ROOT), f"Not found: {DATA_ROOT}"

data_cfg = {
    "path":  DATA_ROOT,
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc": 3,
    "names": ["feeder", "hen", "waterer"],
}
DATA_YAML = os.path.join(DATA_ROOT, "data_local.yaml")
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print(open(DATA_YAML).read())

path: c:\dev\poultry-vision\dataset\segment\merged_poultry_dataset
train: train/images
val: valid/images
test: test/images
nc: 3
names:
- feeder
- hen
- waterer



In [5]:
# Cell 3 — Sanity check: image/label counts + confirm polygon (segmentation) labels
import glob
for split in ["train", "valid", "test"]:
    imgs = glob.glob(os.path.join(DATA_ROOT, split, "images", "*"))
    lbls = glob.glob(os.path.join(DATA_ROOT, split, "labels", "*.txt"))
    print(f"{split:5s}  images={len(imgs):5d}  labels={len(lbls):5d}")

sample = glob.glob(os.path.join(DATA_ROOT, "train", "labels", "*.txt"))[0]
line = open(sample).readline().split()
print("\nSample:", os.path.basename(sample),
      "| class", line[0], "| coords", len(line) - 1,
      "->", "polygon/segmentation" if len(line) > 5 else "bbox/detection")

train  images=  662  labels=  662
valid  images=  142  labels=  142
test   images=  143  labels=  143

Sample: merged_train_0000_cam_low_20251218_151625_frame_7997_jpg.rf.1e507b0cc6477d2f03f5fcb8acca3816.txt | class 1 | coords 36 -> polygon/segmentation


In [6]:
# Cell 5 — Shared config + reusable trainer
from ultralytics import YOLO

COMMON = dict(
    data=DATA_YAML,
    epochs=300,
    patience=50,
    imgsz=640,
    batch=32,           # RTX 5070 12 GB GDDR7 — safe at 32; bump to 48 if no OOM
    device=0,
    seed=SEED,
    deterministic=True,
    cache="ram",        # local machine: load dataset into RAM, faster epochs
    workers=4,          # Windows: 4 is stable; 0 or 2 causes slow DataLoader
    close_mosaic=10,
    plots=True,
    val=True,
)


def run(weights, name, extra=None):
    args = {**COMMON, "name": name, **(extra or {})}
    print(f"\n=== {name}  <-  {weights} ===")
    model = YOLO(weights)                 # task inferred from the weights/yaml
    return model.train(**args)

In [ ]:
# Cell 6 — Object detection (both fine-tune from pretrained). Run one, let it finish, then the next.
run("yolov8s.pt", "det_yolov8s")


=== det_yolov8s  <-  yolov8s.pt ===
New https://pypi.org/project/ultralytics/8.4.70 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.66  Python-3.13.14 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\dev\poultry-vision\dataset\segment\merged_poultry_dataset\data_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300,

c:\dev\poultry-vision\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 0.00.0 ms, read: 327.4117.3 MB/s, size: 93.9 KB)
val: Scanning C:\dev\poultry-vision\dataset\segment\merged_poultry_dataset\valid\labels... 142 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 142/142 658.8it/s 0.2s0.0s
val: New cache created: C:\dev\poultry-vision\dataset\segment\merged_poultry_dataset\valid\labels.cache
WARNING cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.
val: Caching images (0.2GB RAM): 100% ━━━━━━━━━━━━ 142/142 3.0Kit/s 0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001429, momentum=0.9) with parameter

RuntimeError: DataLoader worker (pid(s) 1652) exited unexpectedly

In [9]:
run("yolo12s.pt", "det_yolo12s")


=== det_yolo12s  <-  yolo12s.pt ===
New https://pypi.org/project/ultralytics/8.4.70 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.66  Python-3.13.14 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\dev\poultry-vision\dataset\segment\merged_poultry_dataset\data_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000025F4E307E70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          

In [7]:
# Cell 7 — Instance segmentation
run("yolov8s-seg.pt", "seg_yolov8s")                       # COCO-pretrained -> fine-tune



=== seg_yolov8s  <-  yolov8s-seg.pt ===
New https://pypi.org/project/ultralytics/8.4.71 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.66  Python-3.13.14 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\dev\poultry-vision\dataset\segment\merged_poultry_dataset\data_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=

c:\dev\poultry-vision\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 0.00.0 ms, read: 221.480.4 MB/s, size: 93.9 KB)
val: Scanning C:\dev\poultry-vision\dataset\segment\merged_poultry_dataset\valid\labels.cache... 142 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 142/142 59.6Mit/s 0.0s
WARNING cache='ram' may produce non-deterministic training results. Consider cache='disk' as a deterministic alternative if your disk space allows.
val: Caching images (0.2GB RAM): 100% ━━━━━━━━━━━━ 142/142 3.1Kit/s 0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001429, momentum=0.9) with parameter groups 66 weight(decay=0.0), 77 weight(decay=0.0005), 76 bias(decay=0.0)
Plotting labels to C:\dev\pou

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002CAD9049470>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    

In [8]:
run("yolo12s-seg.yaml", "seg_yolo12s",                     # NO pretrained .pt -> from scratch
    extra=dict(pretrained=False))


=== seg_yolo12s  <-  yolo12s-seg.yaml ===
New https://pypi.org/project/ultralytics/8.4.71 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.66  Python-3.13.14 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\dev\poultry-vision\dataset\segment\merged_poultry_dataset\data_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_de

RuntimeError: bad allocation

In [ ]:
# Cell 8 — Best-weights metrics on the held-out TEST split (box + mask mAP)
import pandas as pd
runs = {"det_yolov8s": "detect", "det_yolo12s": "detect",
        "seg_yolov8s": "segment", "seg_yolo12s": "segment"}
rows = []
for name, task in runs.items():
    best = os.path.join(PROJECT, name, "weights", "best.pt")
    if not os.path.exists(best):
        continue
    m = YOLO(best).val(data=DATA_YAML, split="test", verbose=False)
    row = {"run": name, "task": task,
           "box_mAP50": round(m.box.map50, 4), "box_mAP50-95": round(m.box.map, 4)}
    if task == "segment":
        row["mask_mAP50"] = round(m.seg.map50, 4)
        row["mask_mAP50-95"] = round(m.seg.map, 4)
    rows.append(row)
df = pd.DataFrame(rows); df

In [ ]:
# Cell 9 — Visual spot-check on a few test images
best = os.path.join(PROJECT, "seg_yolov8s", "weights", "best.pt")
test_imgs = sorted(glob.glob(os.path.join(DATA_ROOT, "test", "images", "*")))[:6]
YOLO(best).predict(test_imgs, imgsz=640, conf=0.25, save=True,
                   project=PROJECT, name="preds", exist_ok=True)
print("Annotated images ->", os.path.join(PROJECT, "preds"))

In [ ]:
YOLO(os.path.join(PROJECT, "det_yolov8s", "weights", "last.pt")).train(resume=True)